# Hybrid Undersampling and Oversampling for Handling Imbalanced Credit Card Data
**Implementation of the Research Paper by Maram Alamri and Mourad Ykhlef (IEEE Access, 2024)**

---

### Abstract Overview
Credit card fraud detection suffers from extreme class imbalance (less than 0.13% fraud transactions). Conventional oversampling and undersampling techniques face limitations such as noise generation, overlapping class boundaries, and overfitting.

This project implements the proposed hybrid framework:
1. **Tomek Links Undersampling (Algorithm 1)**: Identifies and removes borderline majority instances (noise removal).
2. **BIRCH Clustering Borderline SMOTE - BCBSMOTE (Algorithms 2 & 3)**:
   - Clusters minority instances with BIRCH to capture local data geometry.
   - Eliminates noise clusters (< 5 points).
   - Identifies borderline 'Danger' minority points.
   - Generates synthetic samples locally and proportionally within clusters using Borderline-SMOTE.
3. **Random Forest Classifier**: Trained on balanced data and evaluated on held-out test transactions.


## 1. Setup & Environment
Import core libraries and project modules from `src/`.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_raw_dataset, clean_and_verify_dataset, preprocess_features, prepare_train_test_split
from src.sampling import TomekLinksUndersampler, BCBSMOTEOversampler, HybridTomekBCBSMOTE
from src.model import FraudDetectionModel
from src.visualization import (
    plot_class_distribution,
    plot_correlation_matrix,
    plot_transaction_type_distribution,
    plot_fraud_by_transaction_type,
    plot_precision_recall_curve,
    plot_confusion_matrix_heatmap
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup completed successfully!')


## 2. Dataset Loading & Exploratory Data Analysis (EDA)
We load the PaySim synthetic credit card transaction dataset.


In [ ]:
DATA_PATH = '../archive/PS_20174392719_1491204439457_log.csv'

# Load a sample for fast notebook execution (set sample_size=None for full 6.36M dataset)
df_sample = load_raw_dataset(DATA_PATH, sample_size=200000, random_state=42)
df_clean = clean_and_verify_dataset(df_sample)
df_clean.head()


### 2.1 Class Skewness (Figure 1 in Paper)
Notice the immense disparity between genuine and fraudulent transactions.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
counts = df_clean['isFraud'].value_counts()
ax.bar(['0 (Normal)', '1 (Fraud)'], counts.values, color=['#4CAF50', '#8BC34A'], width=0.45)
ax.set_yscale('log')
ax.set_ylabel('Number of Transactions (Log Scale)')
ax.set_title('Figure 1: Numbers of Normal and Fraud Transactions in PaySim Dataset')
for i, v in enumerate(counts.values):
    ax.text(i, v * 1.1, f'{v:,}', ha='center', fontweight='bold')
plt.show()


### 2.2 Feature Correlation Matrix (Figure 2 in Paper)
Notice that `newbalanceOrig` and `oldbalanceOrg` have an almost perfect correlation (~1.0).
Per Section III.B of the paper, `newbalanceOrig` and `isFlaggedFraud` are dropped during preprocessing.


In [ ]:
corr_cols = [c for c in ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 
                         'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'] if c in df_clean.columns]
corr = df_clean[corr_cols].corr()

plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', vmin=-1.0, vmax=1.0)
plt.title('Figure 2: Correlation Matrix for PaySim Dataset')
plt.show()


### 2.3 Transaction Types & Fraud Distribution (Figures 3 & 4 in Paper)
- `CASH_OUT` and `PAYMENT` are the most frequent transaction types.
- Fraud ONLY occurs in `CASH_OUT` and `TRANSFER` operations!


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Figure 3: Pie chart
type_counts = df_clean['type'].value_counts()
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.2f%%', startangle=140,
            colors=['#8BC34A', '#29B6F6', '#FFCA28', '#AB47BC', '#26A69A'])
axes[0].set_title('Figure 3: Proportions of Transaction Types')

# Figure 4: Bar chart by fraud status
grouped = df_clean.groupby(['type', 'isFraud']).size().unstack(fill_value=0)
grouped.columns = ['Genuine', 'Fraud']
grouped.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#F44336'], width=0.6)
axes[1].set_yscale('log')
axes[1].set_ylabel('Number of Transactions (Log scale)')
axes[1].set_title('Figure 4: Fraud vs Genuine Count by Transaction Type')
axes[1].set_xticklabels(grouped.index, rotation=0)

plt.tight_layout()
plt.show()


## 3. Data Preprocessing & Stratified Train/Test Split
We drop collinear features (`newbalanceOrig`), heuristic flag (`isFlaggedFraud`), and ID strings (`nameOrig`, `nameDest`),
encode transaction `type`, and split 80% Train / 20% Test (Figure 6 in the paper).


In [ ]:
X, y, label_encoder = preprocess_features(df_clean)
X_train, X_test, y_train, y_test = prepare_train_test_split(X, y, test_size=0.20, random_state=42)
print('Retained Feature Columns:', list(X.columns))


## 4. Hybrid Resampling Methodology
### 4.1 Tomek Links Undersampling (Algorithm 1)
Removes majority points that form mutual nearest neighbor links with minority points to clean noise along the decision boundary.


In [ ]:
tl = TomekLinksUndersampler()
X_tl, y_tl = tl.fit_resample(X_train, y_train)


### 4.2 BIRCH Clustering Borderline SMOTE (Algorithms 2 & 3)
1. Clusters minority samples with BIRCH.
2. Discards noise clusters (< 5 samples).
3. Evaluates borderline Danger samples.
4. Synthesizes new minority points within local clusters.


In [ ]:
bcbsmote = BCBSMOTEOversampler(cluster_size=5, sampling_ratio=0.02, random_state=42)
X_res, y_res = bcbsmote.fit_resample(X_tl, y_tl)


## 5. Model Training (Random Forest Classifier)
Train Random Forest on the cleaned and balanced training set.


In [ ]:
model = FraudDetectionModel(
    sampler=None, # Training set is already resampled
    n_estimators=40,
    max_depth=15,
    random_state=42
)
model.fit(X_res, y_res, resample=False)


## 6. Performance Evaluation on Test Set
Compute Accuracy, Precision, Recall, F1-Score, AUPRC, and Confusion Matrix (Tables 4 & 5).


In [ ]:
# Optimize threshold for F1-score
model.optimize_threshold(X_train, y_train, metric='f1')

# Evaluate on held-out test data
metrics = model.evaluate(X_test, y_test)


### 6.1 Precision-Recall Curve (Figure 10) & Confusion Matrix (Table 5)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Precision-Recall Curve (Figure 10)
from sklearn.metrics import precision_recall_curve
precision, recall, _ = precision_recall_curve(y_test, metrics['Probabilities'])
axes[0].plot(recall, precision, color='#1E88E5', lw=2.5, label=f"Proposed (AUPRC = {metrics['AUPRC']:.4f})")
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f"Figure 10: Precision-Recall Curve (AUPRC = {metrics['AUPRC']*100:.2f}%)")
axes[0].legend(loc='lower left')

# Confusion Matrix Heatmap (Table 5)
cm = metrics['Confusion_Matrix']
labels = np.array([
    [f"TN\n{cm[0, 0]:,}", f"FP\n{cm[0, 1]:,}"],
    [f"FN\n{cm[1, 0]:,}", f"TP\n{cm[1, 1]:,}"]
])
sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False, ax=axes[1],
            xticklabels=['Pred Normal (0)', 'Pred Fraud (1)'],
            yticklabels=['Actual Normal (0)', 'Actual Fraud (1)'])
axes[1].set_title('Table 5: Confusion Matrix for Test Set')

plt.tight_layout()
plt.show()


## 7. Real-Time Transaction Scoring & Inference
Test the trained model on sample transactions.


In [ ]:
# Example: High-risk transfer transaction
test_transaction = pd.DataFrame([{
    'step': 1,
    'type': label_encoder.transform(['TRANSFER'])[0],
    'amount': 250000.0,
    'oldbalanceOrg': 250000.0,
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0
}])

prob = model.predict_proba(test_transaction)[0]
pred = model.predict(test_transaction)[0]

print('=' * 45)
print('TRANSACTION INFERENCE RESULT')
print('=' * 45)
print(f'Fraud Probability : {prob * 100:.2f}%')
print(f'Prediction        : {"[ALERT: FRAUD DETECTED]" if pred == 1 else "[LEGITIMATE]"}')
print('=' * 45)
